In [258]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import xml.etree.ElementTree as ET
from PIL import Image
import torchvision.transforms as transforms 
from sklearn.model_selection import train_test_split

In [259]:
def parse_all_xmls(xml_folder_path):
    all_image_data = []
    
    for filename in os.listdir(xml_folder_path):
        if not filename.endswith(".xml"):
            continue 
            
        xml_path = os.path.join(xml_folder_path, filename)
        
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        size = root.find("size")
        img_width = float(size.find("width").text)
        img_height = float(size.find("height").text)
    
        boxes = []
        labels = []
        
        for obj in root.findall("object"):
            label = obj.find("name").text
            
            bndbox = obj.find("bndbox")
            xmin = float(bndbox.find("xmin").text)
            ymin = float(bndbox.find("ymin").text)
            xmax = float(bndbox.find("xmax").text)
            ymax = float(bndbox.find("ymax").text)
            
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(label)
            
        image_data = {
            "image_filename": filename.replace(".xml", ".jpg"), 
            "width": img_width,
            "height": img_height,
            "boxes": boxes,
            "labels": labels
        }
        
        all_image_data.append(image_data)
        
    return all_image_data

In [260]:
all_parsed_data = parse_all_xmls("../data/annotations/xmls/")

all_parsed_data[:2]

[{'image_filename': 'British_Shorthair_10.jpg',
  'width': 233.0,
  'height': 350.0,
  'boxes': [[83.0, 29.0, 197.0, 142.0]],
  'labels': ['cat']},
 {'image_filename': 'german_shorthaired_119.jpg',
  'width': 500.0,
  'height': 500.0,
  'boxes': [[128.0, 22.0, 240.0, 222.0]],
  'labels': ['dog']}]

In [261]:
stratify_labels = [" ".join(item["image_filename"].split("_")[:-1]) for item in all_parsed_data]

stratify_labels[:5]

['British Shorthair',
 'german shorthaired',
 'english setter',
 'Siamese',
 'pomeranian']

In [262]:
train_data, temp_data, _, temp_labels = train_test_split(
    all_parsed_data, 
    stratify_labels, 
    test_size=0.20, 
    stratify=stratify_labels, 
    random_state=42 
)

len(train_data), len(temp_data)

(2948, 738)

In [263]:
val_data, test_data = train_test_split(
    temp_data, 
    test_size=0.50, 
    stratify=temp_labels, 
    random_state=42
)

len(val_data), len(test_data)

(369, 369)

In [264]:
print(f"Total Images: {len(all_parsed_data)}")
print(f"Train Set: {len(train_data)}")
print(f"Validation Set: {len(val_data)}")
print(f"Test Set: {len(test_data)}")

Total Images: 3686
Train Set: 2948
Validation Set: 369
Test Set: 369


In [265]:
class CatDogDataset(Dataset):
    def __init__(self, parsed_data_list, img_folder_path, S, B, img_width, img_height):
        self.S = S
        self.B = B
        self.img_width = img_width
        self.img_height = img_height
        self.img_folder_path = img_folder_path

        self.parsed_data = parsed_data_list
        self.classes = self._create_classes()

        self.transform = transforms.Compose([
            transforms.Resize((self.img_width, self.img_height)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.parsed_data)

    def __getitem__(self, index):
        item = self.parsed_data[index]
        
        img = Image.open(self.img_folder_path + item["image_filename"]).convert("RGB")

        data = self.transform(img)

        original_width = float(item["width"])
        original_height = float(item["height"])
        
        new_height = data.shape[1] 
        new_width = data.shape[2]

        height_scaling = new_height / original_height
        width_scaling = new_width / original_width

        new_boxes = []
        for original_box in item["boxes"]:
            xmin = original_box[0] * width_scaling
            ymin = original_box[1] * height_scaling
            xmax = original_box[2] * width_scaling
            ymax = original_box[3] * height_scaling

            new_boxes.append([xmin, ymin, xmax, ymax])

        breed_name = " ".join(item["image_filename"].split("_")[:-1])
        labels = [breed_name] * len(new_boxes)

        targets = {
            "labels": [self.classes[label] for label in labels],
            "bbox": new_boxes
        }

        targets = self._create_target_tensor(targets)

        return data, targets
        
    def _create_target_tensor(self, targets):
        out_dim = len(self.classes) + self.B * 5
        output = torch.zeros((self.S, self.S, out_dim))
    
        for bbox, label in zip(targets["bbox"], targets["labels"]):
            bbox_x_min, bbox_y_min, bbox_x_max, bbox_y_max = bbox

            bbox_width = bbox_x_max - bbox_x_min
            bbox_height = bbox_y_max - bbox_y_min

            bbox_center_x = bbox_x_min + bbox_width / 2.0
            bbox_center_y = bbox_y_min + bbox_height / 2.0

            grid_cell_width = self.img_width / self.S
            grid_cell_height = self.img_height / self.S

            grid_x_coord = int(bbox_center_x / grid_cell_width)
            grid_y_coord = int(bbox_center_y / grid_cell_height)
            
            grid_x_coord = min(grid_x_coord, self.S - 1)
            grid_y_coord = min(grid_y_coord, self.S - 1)

            new_bbox_width = bbox_width / self.img_width
            new_bbox_height = bbox_height / self.img_height
            new_bbox_x = (bbox_center_x / grid_cell_width) - grid_x_coord
            new_bbox_y = (bbox_center_y / grid_cell_height) - grid_y_coord
            conf = 1.0

            output[grid_y_coord, grid_x_coord, 0] = new_bbox_x
            output[grid_y_coord, grid_x_coord, 1] = new_bbox_y
            output[grid_y_coord, grid_x_coord, 2] = new_bbox_width
            output[grid_y_coord, grid_x_coord, 3] = new_bbox_height
            output[grid_y_coord, grid_x_coord, 4] = conf
            
            output[grid_y_coord, grid_x_coord, self.B * 5 + label] = 1.0 

        return output
    
    def _create_classes(self):
        with open("../data/annotations/list.txt", "r") as f:
            text = f.read()

        classes = set([" ".join(sent.split(" ")[0].split("_")[:-1]) for sent in text.split("\n")])

        classes.discard('')

        sorted_classes = sorted(list(classes))

        class_to_id = {breed: index for index, breed in enumerate(sorted_classes)}

        return class_to_id

In [266]:
xml_folder_path = "../data/annotations/xmls"
img_folder_path = "../data/images/"
S = 7
B = 2
img_width = 448
img_height = 448

train_dataset = CatDogDataset(train_data, "../data/images/", S, B, img_width, img_height)
val_dataset = CatDogDataset(val_data, "../data/images/", S, B, img_width, img_height)
test_dataset = CatDogDataset(test_data, "../data/images/", S, B, img_width, img_height)

In [267]:
batch_size = 64

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
)

In [269]:
images, targets = next(iter(train_loader))

images.shape, targets.shape

(torch.Size([64, 3, 448, 448]), torch.Size([64, 7, 7, 47]))